# matvec — ex2: batched matvec two ways: torch.bmm with unsqueeze/squeeze vs einsum 'bij,bj->bi'

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `matvec`. Running the final beacon cell reports progress against the `PyTorch: matrix-vector product` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: matrix-vector product` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`matvec`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "matvec"
DD_SUBTOPIC = "PyTorch: matrix-vector product"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Batched matvec — `bmm` vs `einsum('bij,bj->bi', A, x)`

Ex1 did single-sample matvec `W @ x`. The deepening move is the BATCHED variant: `(B, M, N) @ (B, N) → (B, M)`. Two equivalent expressions:

```python
# Option 1: torch.bmm — but bmm wants (B, M, N) @ (B, N, 1) → (B, M, 1)
y_bmm = t.bmm(A, x.unsqueeze(-1)).squeeze(-1)   # (B, M)

# Option 2: einsum — direct shape declaration, no reshape
y_einsum = t.einsum('bij,bj->bi', A, x)         # (B, M)
```

**`bmm` requires 3-D × 3-D.** You can't do `bmm(A, x)` when `x` is 2-D `(B, N)` — bmm strictly requires `(B, N, K)` on the right and produces `(B, M, K)`. The matvec emerges as the `K=1` special case with explicit `unsqueeze` + `squeeze`.

**einsum just declares the shape.** `'bij,bj->bi'` says: batch shared on `b`; contract over `j`; output is `(b, i)`. No reshape scaffolding — the contraction pattern IS the function signature.

**Why both are useful to know.** `bmm` is a single fast kernel — good for inner loops. `einsum` is readable for diverse contraction patterns. Production code mixes them: bmm for hot paths, einsum for one-off transformations.

**Numerical equivalence.** Both compute the same FLOPs in the same order — `allclose` with default tolerance succeeds. Differences only appear with mixed dtypes (`bmm` may downcast intermediate accumulators on some backends; einsum has its own rules).

### Exercise 2 — batched matvec two ways: torch.bmm with unsqueeze/squeeze vs einsum 'bij,bj->bi'

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply two equivalent batched-matvec expressions — `torch.bmm(A, x.unsqueeze(-1)).squeeze(-1)` and `torch.einsum('bij,bj->bi', A, x)` — that both turn `(B, M, N) × (B, N) → (B, M)`, then verify they agree numerically.
> Keywords: batched-matvec, bmm, einsum, shape-discipline
> ```

**KCs targeted:** `bmm-rank3-by-rank3-only`, `einsum-shape-declaration`

Implement `ex2_batched_matvec(A, x)`.

Inputs:
- `A`: `(B, M, N)` float tensor.
- `x`: `(B, N)` float tensor.

Compute the batched matrix-vector product TWO WAYS and return both, plus a numerical equality flag:
```
{
  'y_bmm': (B, M),     # torch.bmm(A, x.unsqueeze(-1)).squeeze(-1)
  'y_einsum': (B, M),  # torch.einsum('bij,bj->bi', A, x)
  'allclose': bool,    # True — both methods agree (atol=1e-6)
}
```

Constraints:
1. `y_bmm` MUST use `torch.bmm` (not `matmul`, not `@`). The drill is about the rank-3 requirement of `bmm`.
2. `y_einsum` MUST use `torch.einsum` with the exact spec string `'bij,bj->bi'`.
3. DO NOT use a Python for-loop.
4. Output dtype matches input.
5. Both outputs are shape `(B, M)` — NOT `(B, M, 1)`.

In [ ]:
def ex2_batched_matvec(A, x):
    y_bmm = t.bmm(A, x.unsqueeze(-1)).squeeze(-1)
    y_einsum = t.einsum('bij,bj->bi', A, x)
    return {
        'y_bmm': y_bmm,
        'y_einsum': y_einsum,
        'allclose': bool(t.allclose(y_bmm, y_einsum, atol=1e-6)),
    }


<details><summary>Solution</summary>

```python
def ex2_batched_matvec(A, x):
    y_bmm = t.bmm(A, x.unsqueeze(-1)).squeeze(-1)
    y_einsum = t.einsum('bij,bj->bi', A, x)
    return {
        'y_bmm': y_bmm,
        'y_einsum': y_einsum,
        'allclose': bool(t.allclose(y_bmm, y_einsum, atol=1e-6)),
    }
```

**`bmm` is the rank-3 by rank-3 batched matmul.** It rejects rank-2 inputs. The `unsqueeze(-1)` adds a trailing dim of 1 to `x` (turning `(B, N)` into `(B, N, 1)`) so the contraction shapes line up: `(B, M, N) @ (B, N, 1) → (B, M, 1)`. The trailing `squeeze(-1)` removes that artificial axis.

**einsum declares shape via the spec string.** `'bij,bj->bi'` reads: 'A has axes b, i, j; x has axes b, j; output has b, i'. The shared `b` is broadcast/batched. The shared `j` is contracted (summed over). The unique `i` becomes the output axis. No reshape scaffolding required.

**Why both — pick by context.** `bmm` is a single fast kernel in CUDA — preferred for hot training loops. `einsum` is more expressive — preferred for one-off transformations where readability matters more than the last 5% perf. Both compile to the same hardware kernel under modern PyTorch.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()